# One phone company needs to know if person will leave them or not

In [39]:
import pandas as pd
import numpy as np


In [40]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')


print(df.head())
print(df.shape) # (7043, 21)
print(df.info())
print(df.describe())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [41]:
pd.crosstab(df['Contract'], df['Churn'], normalize='index')

Churn,No,Yes
Contract,,
Month-to-month,0.572903,0.427097
One year,0.887305,0.112695
Two year,0.971681,0.028319


In [42]:
mask = df['TotalCharges'].str.strip() == ''
df.loc[mask, 'TotalCharges'] = np.nan
data = df.dropna(subset=['TotalCharges'])
print(df.shape)

(7043, 21)


## Data cleaning and feature engineering

In [43]:
# 1. Drop customer ID
df = df.drop(columns=["customerID"])

# 2. Fix TotalCharges
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].replace(r"^\s*$", pd.NA, regex=True),
    errors="coerce"
)

# 3. Handle missing values
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# 4. Encode binary categorical columns

binary_cols = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
    "Churn"
]

for col in binary_cols:
    if col == "gender":
        df[col] = df[col].map({"Female": 0, "Male": 1})
    else:
        df[col] = df[col].map({"No": 0, "Yes": 1})

# 5. One-hot encode multi-category columns
categorical_cols = [
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod"
]

df = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

print(df.head())
print(df.shape)

   gender  SeniorCitizen  Partner  Dependents  tenure  PhoneService  \
0       0              0        1           0       1             0   
1       1              0        0           0      34             1   
2       1              0        0           0       2             1   
3       1              0        0           0      45             0   
4       0              0        0           0       2             1   

   PaperlessBilling  MonthlyCharges  TotalCharges  Churn  ...  \
0                 1           29.85         29.85      0  ...   
1                 0           56.95       1889.50      0  ...   
2                 1           53.85        108.15      1  ...   
3                 0           42.30       1840.75      0  ...   
4                 1           70.70        151.65      1  ...   

   TechSupport_Yes  StreamingTV_No internet service  StreamingTV_Yes  \
0                0                                0                0   
1                0                    

## Logistic Regression

In [44]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X = df.drop(columns='Churn')
y = df['Churn']

num_cols = [
    'SeniorCitizen',
    'tenure',
    'MonthlyCharges',
    'TotalCharges'
]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.fit_transform(X_test[num_cols])

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
print(model.score(X_test, y_test))

0.8048261178140526


In [45]:
from sklearn.metrics import classification_report, confusion_matrix
print(confusion_matrix(y_test, model.predict(X_test)))
print(classification_report(y_test, model.predict(X_test)))

[[929 106]
 [169 205]]
              precision    recall  f1-score   support

           0       0.85      0.90      0.87      1035
           1       0.66      0.55      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.80      0.80      0.80      1409



# decision tree

In [46]:
from sklearn.tree import DecisionTreeClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)
print(model.score(X_test, y_test)) # 0.794180269694819
print(model.score(X_train, y_train))

0.7955997161107168
0.7925097621583245


## KNN

In [47]:
from sklearn.neighbors import KNeighborsClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.fit_transform(X_test[num_cols])

knn = KNeighborsClassifier(n_neighbors=20)
knn.fit(X_train, y_train)
print(knn.score(X_test, y_test))
print(knn.score(X_train, y_train))

0.7863733144073811
0.8146964856230032


## Random Forest

In [48]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(max_depth=8, n_estimators=200 ,random_state=42)
rf.fit(X_train, y_train)

print(rf.score(X_test, y_test))
print(rf.score(X_train, y_train))

print(classification_report(y_test, rf.predict(X_test)))

0.7977288857345636
0.8301384451544196
              precision    recall  f1-score   support

           0       0.82      0.92      0.87      1035
           1       0.68      0.46      0.55       374

    accuracy                           0.80      1409
   macro avg       0.75      0.69      0.71      1409
weighted avg       0.78      0.80      0.78      1409



## SVM

In [49]:
from sklearn.svm import SVC

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.fit_transform(X_test[num_cols])

svm = SVC(kernel='rbf',C=1.0, gamma='scale' ,random_state=42)
svm.fit(X_train, y_train)
pred = svm.predict(X_test)
print(len(svm.support_))

2620


In [51]:
from sklearn.model_selection import GridSearchCV

params = {
    'C': [0.1, 1, 10],
    'gamma' : ['scale', 0.01, 0.1],
}

grid = GridSearchCV(SVC(), params, cv=5, scoring='recall', n_jobs=-1)
grid.fit(X_train, y_train)

print(grid.score(X_test, y_test))
print(grid.score(X_train, y_train))

print(classification_report(y_test, grid.predict(X_test)))

# recall pass betda shunga svm ishlatmimiz

0.5187165775401069
0.6341137123745819
              precision    recall  f1-score   support

           0       0.84      0.88      0.86      1035
           1       0.61      0.52      0.56       374

    accuracy                           0.78      1409
   macro avg       0.72      0.70      0.71      1409
weighted avg       0.78      0.78      0.78      1409

